# Stage 3: Agent Architectures Comparison

This notebook compares 3 agent architectures on the same claims with the same retrieval:
- **Single-pass**: All evidence + single LLM call → verdict
- **LangGraph multi-agent**: 4-node graph (Parse → Retrieve → Review → Verdict)
- **Rerouting (adaptive)**: LangGraph with loop-back when evidence is weak

All use `semantic` chunking + `hybrid_reranked` retrieval + `claude-sonnet-4`.

We measure: verdict accuracy, explanation length, latency, and qualitative explanation quality.

In [ ]:
import json
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

with open("data/test_claims.json") as f:
    claims = json.load(f)

print(f"Test claims: {len(claims)}")

## 3.1 Run single-pass (E2 config)

In [ ]:
from src.pipelines.configurable import run_experiment

single_pass_results = []
for c in claims:
    print(f"  {c['claim'][:50]}...", end=" ", flush=True)
    result = run_experiment(
        c["claim"],
        chunking_strategy="semantic",
        retrieval_method="hybrid_reranked",
        agent_architecture="single_pass",
        model="claude-sonnet-4",
    )
    result["expected_verdict"] = c["expected_verdict"]
    result["correct"] = result["verdict"] == c["expected_verdict"]
    single_pass_results.append(result)
    print(f"→ {result['verdict']} ({'OK' if result['correct'] else 'WRONG'}) {result['metadata']['latency_seconds']}s")

sp_correct = sum(1 for r in single_pass_results if r["correct"])
print(f"\nSingle-pass accuracy: {sp_correct}/{len(single_pass_results)}")

## 3.2 Run LangGraph multi-agent (E6 config)

In [ ]:
langgraph_results = []
for c in claims:
    print(f"  {c['claim'][:50]}...", end=" ", flush=True)
    result = run_experiment(
        c["claim"],
        chunking_strategy="semantic",
        retrieval_method="hybrid_reranked",
        agent_architecture="langgraph_multi",
        model="claude-sonnet-4",
    )
    result["expected_verdict"] = c["expected_verdict"]
    result["correct"] = result["verdict"] == c["expected_verdict"]
    langgraph_results.append(result)
    print(f"→ {result['verdict']} ({'OK' if result['correct'] else 'WRONG'}) {result['metadata']['latency_seconds']}s")

lg_correct = sum(1 for r in langgraph_results if r["correct"])
print(f"\nLangGraph accuracy: {lg_correct}/{len(langgraph_results)}")

## 3.3 Run rerouting / adaptive (E7 config)

In [ ]:
rerouting_results = []
for c in claims:
    print(f"  {c['claim'][:50]}...", end=" ", flush=True)
    result = run_experiment(
        c["claim"],
        chunking_strategy="semantic",
        retrieval_method="hybrid_reranked",
        agent_architecture="strands_rerouting",
        model="claude-sonnet-4",
    )
    result["expected_verdict"] = c["expected_verdict"]
    result["correct"] = result["verdict"] == c["expected_verdict"]
    rerouting_results.append(result)
    print(f"→ {result['verdict']} ({'OK' if result['correct'] else 'WRONG'}) {result['metadata']['latency_seconds']}s")

rt_correct = sum(1 for r in rerouting_results if r["correct"])
print(f"\nRerouting accuracy: {rt_correct}/{len(rerouting_results)}")

## 3.4 Verdict comparison table

In [ ]:
comparison = pd.DataFrame([
    {
        "Claim": c["claim"][:45],
        "Expected": c["expected_verdict"],
        "Single-Pass": sp["verdict"],
        "SP OK": "Y" if sp["correct"] else "",
        "LangGraph": lg["verdict"],
        "LG OK": "Y" if lg["correct"] else "",
        "Rerouting": rt["verdict"],
        "RT OK": "Y" if rt["correct"] else "",
    }
    for c, sp, lg, rt in zip(claims, single_pass_results, langgraph_results, rerouting_results)
])
comparison

## 3.5 Latency comparison

In [ ]:
latency_data = pd.DataFrame([
    {
        "Claim": c["claim"][:30],
        "Single-Pass": sp["metadata"]["latency_seconds"],
        "LangGraph": lg["metadata"]["latency_seconds"],
        "Rerouting": rt["metadata"]["latency_seconds"],
    }
    for c, sp, lg, rt in zip(claims, single_pass_results, langgraph_results, rerouting_results)
])

print("Latency (seconds):")
print(latency_data.to_string(index=False))
print(f"\nTotals: SP={latency_data['Single-Pass'].sum():.1f}s  LG={latency_data['LangGraph'].sum():.1f}s  RT={latency_data['Rerouting'].sum():.1f}s")

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(claims))
width = 0.25
ax.bar([i - width for i in x], latency_data["Single-Pass"], width, label="Single-Pass", color="#4C72B0")
ax.bar(x, latency_data["LangGraph"], width, label="LangGraph", color="#DD8452")
ax.bar([i + width for i in x], latency_data["Rerouting"], width, label="Rerouting", color="#C44E52")
ax.set_xticks(x)
ax.set_xticklabels([c["claim"][:20] + "..." for c in claims], rotation=45, ha="right")
ax.set_ylabel("Seconds")
ax.set_title("Latency by Agent Architecture")
ax.legend()
plt.tight_layout()
plt.show()

## 3.6 Explanation length and quality

In [ ]:
# Explanation length comparison
print(f"{'Claim':<35} {'SP Words':>10} {'LG Words':>10} {'RT Words':>10}")
print("-" * 70)
for c, sp, lg, rt in zip(claims, single_pass_results, langgraph_results, rerouting_results):
    sp_len = len(sp["explanation"].split())
    lg_len = len(lg["explanation"].split())
    rt_len = len(rt["explanation"].split())
    print(f"{c['claim'][:35]:<35} {sp_len:>10} {lg_len:>10} {rt_len:>10}")

avg_sp = sum(len(r["explanation"].split()) for r in single_pass_results) / len(single_pass_results)
avg_lg = sum(len(r["explanation"].split()) for r in langgraph_results) / len(langgraph_results)
avg_rt = sum(len(r["explanation"].split()) for r in rerouting_results) / len(rerouting_results)
print(f"\nAvg explanation length: SP={avg_sp:.0f}  LG={avg_lg:.0f}  RT={avg_rt:.0f} words")

## 3.7 Side-by-side explanation samples

In [ ]:
# Show explanations for the most interesting claim (a nuanced one)
idx = next(i for i, c in enumerate(claims) if c["difficulty"] == "nuanced")
claim = claims[idx]["claim"]

print(f"Claim: {claim}")
print(f"Expected: {claims[idx]['expected_verdict']}")
print("=" * 80)

for name, result_list in [("Single-Pass", single_pass_results), ("LangGraph", langgraph_results), ("Rerouting", rerouting_results)]:
    r = result_list[idx]
    status = "CORRECT" if r["correct"] else "WRONG"
    print(f"\n--- {name} [{r['verdict']}] ({status}) ---")
    print(r["explanation"])
    print(f"Evidence cited: {len(r['evidence'])} passages")

## 3.8 Summary

In [ ]:
summary = pd.DataFrame([
    {
        "Architecture": "Single-Pass",
        "Accuracy": f"{sp_correct}/{len(claims)}",
        "Avg Latency (s)": round(latency_data["Single-Pass"].mean(), 1),
        "Total Latency (s)": round(latency_data["Single-Pass"].sum(), 1),
        "Avg Explanation Words": round(avg_sp),
    },
    {
        "Architecture": "LangGraph Multi-Agent",
        "Accuracy": f"{lg_correct}/{len(claims)}",
        "Avg Latency (s)": round(latency_data["LangGraph"].mean(), 1),
        "Total Latency (s)": round(latency_data["LangGraph"].sum(), 1),
        "Avg Explanation Words": round(avg_lg),
    },
    {
        "Architecture": "Rerouting (Adaptive)",
        "Accuracy": f"{rt_correct}/{len(claims)}",
        "Avg Latency (s)": round(latency_data["Rerouting"].mean(), 1),
        "Total Latency (s)": round(latency_data["Rerouting"].sum(), 1),
        "Avg Explanation Words": round(avg_rt),
    },
])
summary

## Key Takeaways

| Architecture | LLM Calls | Strengths | Weaknesses |
|-------------|-----------|-----------|------------|
| Single-pass | 1 | Fast, cheap | No claim decomposition, less nuanced |
| LangGraph multi-agent | 4 | Structured reasoning, richer explanations | 3-4x slower, more expensive |
| Rerouting | 4-6 | Adapts to evidence quality, fills gaps | Slowest, most expensive |

**Next step:** Stage 4 runs the full experiment matrix (E1-E7) to get statistical comparisons.